# FinBERT – Test de 5 noticias (Clasificación + Summarization + NER)

Con el objetivo de mostrar de forma clara el resultado de los objetivos planteados al inicio del proyecto, se ha desarrollado un script de demostración que permite visualizar el funcionamiento conjunto del sistema sobre noticias reales del conjunto de test. En este script se integran los distintos modelos trabajados a lo largo del proyecto, aplicando sobre cada noticia la clasificación temática mediante el modelo FinBERT ajustado a nuestro dominio, así como los módulos de resumen automático y reconocimiento de entidades (NER). De esta forma, se simula un escenario de uso realista en el que, a partir del texto completo de una noticia financiera, el sistema es capaz de identificar su tópico principal y extraer la información más relevante. Esta demostración no solo permite observar el rendimiento de los modelos desde un punto de vista cuantitativo, sino también analizar de manera cualitativa su comportamiento y coherencia, comprobando que la solución desarrollada cumple con los objetivos iniciales del proyecto y ofrece una base funcional para un sistema de análisis automático de noticias financieras.

- Clasificación de topic (FinBERT)
- Summarization (**placeholder** – lo completa tu compañero)
- NER (**placeholder** – lo completa tu compañero)

Abajo se imprime el texto principal y un "ticker" con: **Resumen**, **Entidades**, **Predicción**, y si la clasificación es **correcta**.

In [11]:
# Imports
import os
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from IPython.display import display, Markdown, HTML

try:
    import joblib
except ImportError:
    joblib = None


## Configuración

In [12]:
# Paths (copiados del notebook anterior)
TEST_CSV = "../data/definitivos/splits/test.csv"
MODEL_DIR ="./Segunda_Resolucion/Text_classification/models/finbert_topicclf"
   # carpeta del modelo fine-tuned
LABEL_ENCODER_PATH = os.path.join(MODEL_DIR, "label_encoder.joblib")  # opcional (recomendado)

# Parámetros
N_SAMPLES = 5
RANDOM_STATE = 42
MAX_LEN = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)


DEVICE: cuda


## Cargar dataset y modelo

In [13]:
import os
os.getcwd()


'c:\\Users\\mpsua\\OneDrive\\Escritorio\\ud\\CUARTO\\Primer_Cuatri\\PLN\\FinTracker\\FinTracker\\src'

In [14]:
# 1) Dataset
df_test = pd.read_csv(TEST_CSV)

if "text_input" not in df_test.columns:
    df_test["text_input"] = df_test["headline"].fillna("") + " " + df_test["summary"].fillna("")

TEXT_COL = "article_text" if "article_text" in df_test.columns else "text_input"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

class_names = None
if joblib is not None and os.path.exists(LABEL_ENCODER_PATH):
    le = joblib.load(LABEL_ENCODER_PATH)
    class_names = list(le.classes_)
else:
    n_classes = int(df_test["label"].max()) + 1 if "label" in df_test.columns else 5
    class_names = [str(i) for i in range(n_classes)]

print("Clases:", class_names)
print("Columnas disponibles:", list(df_test.columns))
print("Texto principal:", TEXT_COL)


Clases: ['Business Growth and Cloud Infrastructure in the AI Industry', 'Financial and Market News and Corporate Sales', 'Informal / Conversational Lenguaje', 'Quantum Computing and Military Technology', 'Stock Market and Trading']
Columnas disponibles: ['ticker', 'headline', 'summary', 'article_text', 'topic', 'text_input', 'label']
Texto principal: article_text


## Funciones: Clasificación + Placeholders para Summarization y NER

In [15]:
def predict_topic(text: str):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()
    pred_id = int(np.argmax(probs))
    pred_name = class_names[pred_id] if pred_id < len(class_names) else str(pred_id)
    return pred_id, pred_name, probs


# ==============================
# Placeholders (lo completará tu compañero)
# ==============================
def summarize_text(text: str) -> str:
    """TODO: implementar summarization"""
    return "[TODO: resumen aquí]"  # placeholder


def extract_entities(text: str):
    """TODO: implementar NER. Devuelve una lista de strings o tu estructura preferida."""
    return ["[TODO: entidades aquí]"]  # placeholder


## Muestreo de 5 noticias del test set

In [16]:
# Seleccionamos 5 noticias aleatorias del test set
samples = df_test.sample(n=min(N_SAMPLES, len(df_test)), random_state=RANDOM_STATE).reset_index(drop=True)
samples[["headline", "topic", "label"]].head() if "topic" in samples.columns and "label" in samples.columns else samples.head()


,headline,topic,label
0,"1 Unstoppable Stock That Could Join Nvidia, Ap...",Business Growth and Cloud Infrastructure in th...,0
1,Oracle Bets Big on Cloud Expansion: A Sign of ...,Business Growth and Cloud Infrastructure in th...,0
2,The 5 Best S&P 500 Stocks of the Last 10 Years,Business Growth and Cloud Infrastructure in th...,0
3,The evolution of cognitive banking,Financial and Market News and Corporate Sales,1
4,Earnings Growth & Price Strength Make Walmart ...,Stock Market and Trading,4


## Visualización + "Ticker" de resultados

Se imprime el texto principal y debajo un bloque tipo ticker:

- **Resumen** (placeholder)
- **Entidades** (placeholder)
- **Clasificación predicha**
- **Correcta** (comparada con `topic` o `label` si existe)


In [17]:
    def render_ticker(summary: str, entities, pred_topic: str, is_correct: bool | None):
        # CSS simple en línea
        correct_text = "N/A" if is_correct is None else ("✅ Correcta" if is_correct else "❌ Incorrecta")
        correct_style = "color:#2e7d32;font-weight:700;" if is_correct else "color:#c62828;font-weight:700;"
        if is_correct is None:
            correct_style = "color:#616161;font-weight:700;"

        entities_str = ", ".join([str(e) for e in entities]) if isinstance(entities, (list, tuple)) else str(entities)

        html = f"""
        <div style="border:1px solid #ddd;border-radius:10px;padding:12px;margin-top:10px;">
          <div style="display:flex;gap:14px;flex-wrap:wrap;">
            <div style="flex:1;min-width:250px;">
              <div style="font-size:12px;color:#555;">Resumen</div>
              <div style="font-size:14px;">{summary}</div>
            </div>
            <div style="flex:1;min-width:250px;">
              <div style="font-size:12px;color:#555;">Entidades</div>
              <div style="font-size:14px;">{entities_str}</div>
            </div>
            <div style="flex:0.8;min-width:220px;">
              <div style="font-size:12px;color:#555;">Clasificación (Topic)</div>
              <div style="font-size:14px;font-weight:700;">{pred_topic}</div>
            </div>
            <div style="flex:0.6;min-width:160px;text-align:right;">
              <div style="font-size:12px;color:#555;">¿Correcta?</div>
              <div style="font-size:14px;{correct_style}">{correct_text}</div>
            </div>
          </div>
        </div>
        """

        return HTML(html)


    for i, row in samples.iterrows():
        headline = str(row.get("headline", f"Noticia {i+1}"))
        true_topic_name = row.get("topic", None)
        true_label = row.get("label", None)

        text = str(row.get(TEXT_COL, ""))
        if not text.strip():
            # fallback por si el campo está vacío
            text = str(row.get("text_input", ""))

        # Predicción
        pred_id, pred_topic, probs = predict_topic(text)

        # Correctitud (si tenemos verdad terreno)
        is_correct = None
        if true_topic_name is not None and isinstance(true_topic_name, str) and true_topic_name.strip():
            is_correct = (pred_topic == true_topic_name)
        elif true_label is not None and not pd.isna(true_label):
            is_correct = (pred_id == int(true_label))

        # Summarization + NER (placeholders)
        summary = summarize_text(text)
        entities = extract_entities(text)

        # Render
        display(Markdown(f"""---
## {i+1}) {headline}
"""))
        display(Markdown(f"""**Texto principal ({TEXT_COL})**"""))
        display(Markdown(text))
        display(render_ticker(summary=summary, entities=entities, pred_topic=pred_topic, is_correct=is_correct))


---
## 1) 1 Unstoppable Stock That Could Join Nvidia, Apple, Microsoft, Amazon, Alphabet, Meta, and Tesla in the $1 Trillion Club


**Texto principal (article_text)**

There are several stocks that are now worth at least $1 trillion. This includes Nvidia, Apple, Microsoft, Amazon, Alphabet, Meta Platforms, and Tesla, which are together known as the "Magnificent Seven." There are others now part of the trillion-dollar club, but these are among the more notable names.
The list may get even bigger in the years ahead, as many businesses are growing and benefiting from feverish spending on artificial intelligence (AI). There's one unstoppable company that's doing well due to that trend, and which looks set to join the club in the near future: Oracle (ORCL 1.45%). Here's why it may not be too late to invest in this promising tech stock.
Oracle has become a key player in the AI boom
For years, Oracle has been known as a big name in databases and providing companies with the critical backend services and infrastructure they need to manage all their data. It still serves that purpose today, plus it also offers key products and services, which help its customers manage AI workloads.
Its new Oracle AI World, for instance, will allow its customers to use large language models to analyze existing databases. The big advantage with Oracle is that unlike many big tech companies, it's agnostic to the whole chatbot market; it isn't developing one of its own, and it's there to simply provide customers with the AI-powered products and services they need. It also has AI agents that can help to automate tasks and add efficiency.
And its efforts have been paying off. Last month, the company reported its latest quarterly results (for the period ending Aug. 31) and demand remains robust, with CEO Safra Catz noting that it "signed four multibillion-dollar contracts with three different customers in Q1." It expects even more large contracts to be signed in the future. And its contract backlog has skyrocketed by 359% year over year, totaling $455 billion.
Oracle's quarterly revenue totaled $14.9 billion and rose 12% year over year. Its cloud segment led the way, as sales in that business unit came in at $7.2 billion and were up by 28%.
The company also generates terrific profit margins, as its net income during the period was $2.9 billion, which is comparable to what it generated in the prior-year period. The bottom line would have been larger, however, if not for an increase in restructuring expenses, as the company has been reducing its workforce in an effort to improve efficiency.
Why a $1 trillion valuation may be just an inevitability at this point
Oracle's market cap as of the end of Monday was around $820 billion. For it to hit a $1 trillion valuation, it would need to rise by another 22% from that. In just the past year, its shares have soared more than 70%. And with tech companies continuing to invest heavily into AI, Oracle looks poised to benefit from those growth opportunities.
It may not happen soon, and it could take a couple of years, as the stock has already risen so fast. It's currently trading around its all-time highs, and its price-to-earnings multiple of 66 doesn't look cheap at all. As a result, growth investors may be thinking twice about the stock, and that could lead to limited gains, at least in the short term.
But with so much growth still out there, it may truly be an inevitability before Oracle reaches $1 trillion. According to analysts from the MarketsandMarkets, the global cloud AI market is projected to grow at a compounded annual growth rate of 32.4% until 2029. It's a staggering level of growth, with Oracle right in the middle of it all.
Oracle's still a good buy today
Although you may be tempted to pass on Oracle's stock because its valuation is a bit rich, over the long term, there's still much more runway for it to become more valuable. The important role it plays in tech makes it a solid investment, particularly due to its close relationship with other tech giants.
Oracle's solid fundamentals and AI-powered growth makes it a great investment to add to your portfolio today, especially if you're looking for an AI stock that still has a lot of upside left.

---
## 2) Oracle Bets Big on Cloud Expansion: A Sign of Strong Upside Ahead?


**Texto principal (article_text)**

...

**Texto principal (article_text)**

Over the past decade, the way people engage with services has changed dramatically. Amazon anticipates what we want to buy, Netflix knows what we’ll watch next, and Spotify curates playlists that feel almost personal. Against this backdrop, it’s no surprise that customers now expect their banks to deliver the same level of convenience, experience, guidance and relevance.
The problem is, banking hasn’t kept pace. Too often, the relationship between a bank and its customer is still defined by products and transactions. The result is a widening gap between what customers need and what they receive, and that’s exactly where disruption takes root.
Today’s customers don’t want apps that simply log their balances or display past spending. They want proactive, intelligent guidance that helps them make decisions in real time. They want their bank to act less like a service provider and more like a financial partner, or advisor, if you will.
What cognitive banking really means
This is where Cognitive Banking comes in. It marks a shift away from superficial “personalisation” toward deep, data-driven engagement. At its heart, Cognitive Banking is about bringing humanity back to digital channels, delivering the kind of financial guidance that was once available only to the wealthy, but now at scale.
By harnessing AI, banks can analyse behaviours, anticipate needs, and step in with meaningful support. That might mean flagging overspending before it spirals, automatically moving idle cash into savings, or nudging someone back on track with their financial goals. The aim isn’t to overwhelm customers with data, but to translate it into timely, actionable insights that genuinely improve their financial wellness.
From engagement to primacy
Engagement alone is no longer enough. According to Accenture’s Global Banking Consumer Study Report, 73% of customers engage with multiple financial institutions beyond their primary bank, and nearly 60% of primary checking accounts churn due to digital competitors. Customers increasingly shop around for the best deal, not the best relationship. For banks, the challenge isn’t just to engage customers, it’s to become their primary financial institution. That requires a more holistic view of wallet share and customer behavior across the entire lifecycle.
US Tariffs are shifting - will you react or anticipate?
Don’t let policy changes catch you off guard. Stay proactive with real-time data and expert analysis.
By GlobalDataCrucially, this means looking beyond a customer’s direct interactions with the bank to understand their broader financial relationships, whether that’s with other providers, apps or payment platforms. Gaining this perspective allows banks to piece together a richer picture of customer needs, leading to more accurate insights, more relevant guidance and ultimately, a stronger relationship. What’s more, it doesn’t require open banking access.
Banks must understand not only how customers use their own products, but also the signals that point to activity elsewhere. With that broader perspective, they can move from offering one-size-fits-all experiences to delivering insights that strengthen loyalty and deepen the relationship.
Innovations now make it possible to segment customers by engagement type and tailor strategies accordingly. For example, new customers in their first 90 days can be nurtured with personalised insights that encourage adoption at the moment when attrition risk is highest. Secondary customers whose activity is drifting can be re-engaged with offers that rebuild trust and capture back wallet share. Customers showing attrition signals can be proactively supported to discourage them from leaving. Each segment receives curated, intelligence-driven guidance that helps them achieve their financial goals while encouraging them to rely more heavily on their primary institution.
Why this matters now
The business case is compelling. Primary customers generate up to 10x more deposits and 8x more fee revenue than non-primary customers. At the same time, research shows that 84% of consumers would consider switching banks to receive more relevant, timely insights. Financial wellness has overtaken health and relationships as a top life priority, and customers are prepared to reward the institutions that help them achieve it.
By harnessing behavioural intelligence and using customer data more intelligently, banks can deliver personalised, contextual guidance, strengthening trust, deepening wallet share, and accelerating the path to primacy.
The road ahead
Cognitive Banking offers more than an upgrade in digital experience; it is a strategy for long-term growth. It gives customers the confidence to make smarter decisions while enabling banks to capture greater loyalty, deposits, and engagement.
The industry is at an inflection point. Customers are demanding more, competition is intensifying, and primacy is the prize. The banks that thrive will be those that combine human understanding with the scale and intelligence of AI to create truly Cognitive Banking experiences; experiences that don’t just engage customers but secure their loyalty for the long term.
Now is the time to act. Customers are ready, the technology is proven, and the path to primacy has never been clearer.
Udi Ziv is CEO, Personetics

---
## 5) Earnings Growth & Price Strength Make Walmart (WMT) a Stock to Watch


**Texto principal (article_text)**

If you're a beginner investor, the idea of creating a portfolio from the ground up can feel like an impossible goal to achieve. That's why you should start by looking at stocks that are set to beat the market over the next 12 months, a strategy that's been proven to generate strong returns.
Now, let's break down why adding this one exceptional stock, highlighted below, to your portfolio could be a recipe for success.
Why You Should Pay Attention to Walmart (WMT)
Walmart Inc. has evolved from just being a traditional brick-and-mortar retailer into an omnichannel player. In this regard, acquisitions; partnerships; delivery programs like Walmart + and Express Delivery; and investment in online e-commerce platform Flipkart are noteworthy. These position the company to keep pace with the changing retail ecosystem and stay firm in the presence of rivals like Amazon and Target. Markedly, Walmart’s product offerings include almost everything from grocery to cosmetics, electronics to stationery, home furnishings to health and wellness products, and apparel to entertainment products, to name a few.
WMT was added to the Zacks Focus List on May 30, 2017 at $26.04 per share. Since then, shares have increased 297.43% to $103.49.
Three analysts revised their earnings estimate upwards in the last 60 days for fiscal 2026. The Zacks Consensus Estimate has increased $0 to $2.6. WMT boasts an average earnings surprise of 2.8%.
Additionally, Walmart's earnings are expected to grow 3.6% for the current fiscal year.
Because stock prices react to revisions, buying stocks with rising earnings estimates can be very profitable. Focus List stocks like WMT offer investors a great opportunity to get into a company whose future earnings estimates will be raised, potentially leading to price momentum.
Want the latest recommendations from Zacks Investment Research? Today, you can download 7 Best Stocks for the Next 30 Days. Click to get this free report
Walmart Inc. (WMT) : Free Stock Analysis Report
This article originally published on Zacks Investment Research (zacks.com).